In [1]:
# ================================================
# Cell 1 — Imports & Model Load
# ================================================
# embedding_dim=128: smaller head keeps the feature space compact for SVM;
# Normalise with ImageNet mean/std because ResNet50 weights are ImageNet-pretrained.
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import resnet50
from PIL import Image
import numpy as np

MODEL_PATH  = "/home/jenarththan/Desktop/FYP/May11/Notebooks/ResNet50_May11_Results/best_resnet50_model.pth"
TEST_ROOT   = "/home/jenarththan/Desktop/FYP/May11/Dataset/Testing"
OUTPUT_DIR  = "/home/jenarththan/Desktop/FYP/May11/Notebooks/ResNet50_May11_Results"
CLASS_NAMES = ['0', '100', '500', '1000', '1500', '2000']
EXTENSIONS  = ('.jpg', '.jpeg', '.png', '.bmp')
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=6, embedding_dim=128, p_drop=0.3):
        super().__init__()
        self.backbone = resnet50(weights=None)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.embedding_head = nn.Sequential(
            nn.Dropout(p_drop),
            nn.Linear(in_features, embedding_dim),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, return_embedding=False):
        features = self.backbone(x)
        emb = self.embedding_head(features)
        logits = self.classifier(emb)
        return (logits, emb) if return_embedding else logits

model = ResNet50Classifier(num_classes=len(CLASS_NAMES), embedding_dim=128)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE).eval()
print(f"Model loaded  |  device = {DEVICE}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
print("Ready.")

Model loaded  |  device = cuda
Ready.


In [2]:
# ================================================
# Cell 2 — Run Inference on all 6 Testing folders
# Builds all_results list used by later cells
# ================================================
import csv

CSV_PATH = os.path.join(OUTPUT_DIR, "inference_probabilities_all_test.csv")
TXT_PATH = os.path.join(OUTPUT_DIR, "inference_probabilities_all_test.txt")
print(f"Results will be saved to:\n  CSV → {CSV_PATH}\n  TXT → {TXT_PATH}\n")

col_probs = '  '.join(f'{c:>7}' for c in CLASS_NAMES)
header    = f"{'Image':<28}  {'True':>5}  {'Pred':>5}  {'Conf%':>6}  {col_probs}  {'Correct':>7}"
divider   = '-' * len(header)

all_results = []   # (fname, true_cls, pred_cls, conf, probs_list, correct)
txt_lines   = []

for true_cls in CLASS_NAMES:
    folder = os.path.join(TEST_ROOT, true_cls)
    images = sorted(f for f in os.listdir(folder) if f.lower().endswith(EXTENSIONS))

    for line in [f"\n{'='*len(header)}", f"  TRUE CLASS: {true_cls}  ({len(images)} images)",
                 f"{'='*len(header)}", header, divider]:
        print(line); txt_lines.append(line)

    folder_correct = 0

    for fname in images:
        img    = Image.open(os.path.join(folder, fname)).convert('RGB')
        tensor = transform(img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            probs = F.softmax(model(tensor), dim=1).squeeze().cpu().numpy()

        pred_cls = CLASS_NAMES[probs.argmax()]
        conf     = probs.max() * 100
        correct  = pred_cls == true_cls
        if correct:
            folder_correct += 1

        prob_str = '  '.join(f'{p*100:>7.2f}' for p in probs)
        tick     = '  YES' if correct else '   NO'
        line     = f"{fname:<28}  {true_cls:>5}  {pred_cls:>5}  {conf:>6.2f}  {prob_str}  {tick}"
        print(line); txt_lines.append(line)

        all_results.append((fname, true_cls, pred_cls, conf, probs.tolist(), correct))

    acc = folder_correct / len(images) * 100
    for line in [divider, f"  Folder accuracy: {folder_correct}/{len(images)}  ({acc:.2f}%)"]:
        print(line); txt_lines.append(line)

total   = len(all_results)
correct = sum(r[5] for r in all_results)
for line in [f"\n{'='*60}",
             f"  OVERALL TEST ACCURACY : {correct}/{total}  ({correct/total*100:.2f}%)",
             f"{'='*60}"]:
    print(line); txt_lines.append(line)

# Save TXT
with open(TXT_PATH, 'w') as f:
    f.write('\n'.join(txt_lines))
print(f"\nTXT saved → {TXT_PATH}")

# Save CSV
csv_cols = ['image', 'true_class', 'pred_class', 'confidence_pct', 'correct'] \
         + [f'prob_class_{c}' for c in CLASS_NAMES]
with open(CSV_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(csv_cols)
    for fname, true_cls, pred_cls, conf, probs_list, correct in all_results:
        writer.writerow(
            [fname, true_cls, pred_cls, f'{conf:.4f}', int(correct)]
            + [f'{p*100:.4f}' for p in probs_list]
        )
print(f"CSV saved → {CSV_PATH}")
print(f"\nall_results ready: {len(all_results)} entries")

Results will be saved to:
  CSV → /home/jenarththan/Desktop/FYP/May11/Notebooks/ResNet50_May11_Results/inference_probabilities_all_test.csv
  TXT → /home/jenarththan/Desktop/FYP/May11/Notebooks/ResNet50_May11_Results/inference_probabilities_all_test.txt


  TRUE CLASS: 0  (500 images)
Image                          True   Pred   Conf%        0      100      500     1000     1500     2000  Correct
-----------------------------------------------------------------------------------------------------------------
10_21.png                         0      0   98.57    98.57     0.05     0.86     0.07     0.44     0.01    YES
10_30.png                         0      0   97.28    97.28     0.03     2.48     0.06     0.15     0.01    YES
10_35.png                         0    500   51.21    47.02     0.10    51.21     1.17     0.48     0.03     NO
10_37.png                         0      0   99.93    99.93     0.00     0.05     0.01     0.01     0.00    YES
10_39.png                         0   

In [3]:
# ================================================
# Cell 3 — Save probability bar chart for every image
# Layout: [aggregate image | horizontal bar chart]
# Output: ResNet50_May11_Results/probability_plots/<class>/<image>.png
# Two-panel layout (image | bar chart) lets the viewer immediately compare
# the aggregate particle with the model confidence — useful for FYP report figures.
# Green = predicted class, blue = others; title turns red on wrong predictions.
# ================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PLOT_ROOT = os.path.join(OUTPUT_DIR, "probability_plots")
for cls in CLASS_NAMES:
    os.makedirs(os.path.join(PLOT_ROOT, cls), exist_ok=True)
print(f"Saving bar chart plots to:\n  {PLOT_ROOT}/\n")

total_saved = 0

for fname, true_cls, pred_cls, conf, probs_list, correct in all_results:

    probs_np = np.array(probs_list)
    pred_idx = probs_np.argmax()

    # Load original aggregate image
    img_pil = Image.open(os.path.join(TEST_ROOT, true_cls, fname)).convert('RGB')

    # Bar colours: green for predicted class, blue for others
    colors      = ['#27AE60' if i == pred_idx else '#4A90D9' for i in range(len(CLASS_NAMES))]
    tick        = 'CORRECT' if correct else 'WRONG'
    title_color = '#27AE60'   if correct else '#E74C3C'

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # ── Left panel: aggregate image ──
    axes[0].imshow(img_pil)
    axes[0].set_title(
        f"True: {true_cls}   Pred: {pred_cls} ({conf:.1f}%)   {tick}",
        fontsize=11, fontweight='bold', color=title_color
    )
    axes[0].axis('off')

    # ── Right panel: horizontal bar chart ──
    bars = axes[1].barh(CLASS_NAMES, probs_np * 100,
                        color=colors, edgecolor='white', height=0.55)
    axes[1].set_xlabel('Probability (%)', fontsize=10)
    axes[1].set_title('Class Probabilities — ResNet50', fontsize=11, fontweight='bold')
    axes[1].set_xlim(0, 108)
    axes[1].invert_yaxis()

    for bar, prob in zip(bars, probs_np):
        axes[1].text(bar.get_width() + 0.8,
                     bar.get_y() + bar.get_height() / 2,
                     f'{prob*100:.2f}%', va='center', fontsize=9)

    green_patch = mpatches.Patch(color='#27AE60', label='Predicted')
    blue_patch  = mpatches.Patch(color='#4A90D9', label='Other')
    axes[1].legend(handles=[green_patch, blue_patch], fontsize=8, loc='lower right')

    plt.tight_layout()

    stem     = os.path.splitext(fname)[0]
    out_path = os.path.join(PLOT_ROOT, true_cls, f"{stem}.png")
    plt.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.close()
    total_saved += 1

    if total_saved % 100 == 0:
        print(f"  Saved {total_saved} / {len(all_results)} ...")

print(f"\nDone.  {total_saved} bar chart images saved to:\n  {PLOT_ROOT}/")

Saving bar chart plots to:
  /home/jenarththan/Desktop/FYP/May11/Notebooks/ResNet50_May11_Results/probability_plots/

  Saved 100 / 3000 ...
  Saved 200 / 3000 ...
  Saved 300 / 3000 ...
  Saved 400 / 3000 ...
  Saved 500 / 3000 ...
  Saved 600 / 3000 ...
  Saved 700 / 3000 ...
  Saved 800 / 3000 ...
  Saved 900 / 3000 ...
  Saved 1000 / 3000 ...
  Saved 1100 / 3000 ...
  Saved 1200 / 3000 ...
  Saved 1300 / 3000 ...
  Saved 1400 / 3000 ...
  Saved 1500 / 3000 ...
  Saved 1600 / 3000 ...
  Saved 1700 / 3000 ...
  Saved 1800 / 3000 ...
  Saved 1900 / 3000 ...
  Saved 2000 / 3000 ...
  Saved 2100 / 3000 ...
  Saved 2200 / 3000 ...
  Saved 2300 / 3000 ...
  Saved 2400 / 3000 ...
  Saved 2500 / 3000 ...
  Saved 2600 / 3000 ...
  Saved 2700 / 3000 ...
  Saved 2800 / 3000 ...
  Saved 2900 / 3000 ...
  Saved 3000 / 3000 ...

Done.  3000 bar chart images saved to:
  /home/jenarththan/Desktop/FYP/May11/Notebooks/ResNet50_May11_Results/probability_plots/


In [4]:
# ================================================
# Cell 4 — Per-class accuracy summary bar chart
# ================================================
# Colour thresholds: green >= 90% (good), orange >= 75% (acceptable), red < 75% (poor).
# Dashed line shows overall accuracy for quick visual comparison per class.
import matplotlib.pyplot as plt

class_correct = {c: 0 for c in CLASS_NAMES}
class_total   = {c: 0 for c in CLASS_NAMES}
for fname, true_cls, pred_cls, conf, probs_list, correct in all_results:
    class_total[true_cls]   += 1
    class_correct[true_cls] += int(correct)

accs = [class_correct[c] / class_total[c] * 100 for c in CLASS_NAMES]
overall_acc = sum(class_correct.values()) / sum(class_total.values()) * 100

colors = ['#27AE60' if a >= 90 else '#F39C12' if a >= 75 else '#E74C3C' for a in accs]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(CLASS_NAMES, accs, color=colors, edgecolor='white', width=0.55)
ax.axhline(overall_acc, color='#2C3E50', linestyle='--', linewidth=1.5,
           label=f'Overall: {overall_acc:.2f}%')
ax.set_xlabel('Milling Class (RPM)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('ResNet50 May11 — Per-Class Test Accuracy', fontsize=13, fontweight='bold')
ax.set_ylim(0, 110)
ax.legend(fontsize=11)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nPer-class breakdown:")
for c in CLASS_NAMES:
    print(f"  Class {c:>4} : {class_correct[c]:>3}/{class_total[c]}  ({class_correct[c]/class_total[c]*100:.2f}%)")
print(f"\nOverall      : {sum(class_correct.values())}/{sum(class_total.values())}  ({overall_acc:.2f}%)")


Per-class breakdown:
  Class    0 : 436/500  (87.20%)
  Class  100 : 482/500  (96.40%)
  Class  500 : 460/500  (92.00%)
  Class 1000 : 351/500  (70.20%)
  Class 1500 : 355/500  (71.00%)
  Class 2000 : 489/500  (97.80%)

Overall      : 2573/3000  (85.77%)


/tmp/ipykernel_5513/2824902351.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
